# Tool Definition Caching — Converse API

This notebook demonstrates how to cache tool (function) definitions for agentic applications using the **Converse** and **ConverseStream** APIs. When you have a large, static set of tools, caching them avoids reprocessing the schemas on every request.

## What you'll learn

- **Converse API**: `cachePoint` as the last element in the `tools` array
- **ConverseStream API**: Same syntax with streaming response parsing

## Tool cache placement

```python
toolConfig = {
    "tools": [
        {"toolSpec": {"name": ..., "inputSchema": {"json": ...}}},
        {"toolSpec": {"name": ..., "inputSchema": {"json": ...}}},
        {"cachePoint": {"type": "default"}}   # after all tools
    ]
}
```

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `boto3 >= 1.42.0`

## Token threshold

Claude Sonnet 4.6 requires at least **2,048 tokens** per cache checkpoint. The 4 tool definitions below collectively exceed this threshold.

In [ ]:
!pip install --upgrade --quiet boto3

In [ ]:
import boto3
import json
import time

MODEL_ID = "global.anthropic.claude-sonnet-4-6"
AWS_REGION = "us-west-2"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Model: {MODEL_ID}")

## Tool Definitions

Space-themed tools with comprehensive schemas.

In [ ]:
TOOLS_LIST = [
    {
        "name": "analyze_celestial_object",
        "description": "Analyzes a celestial object and returns detailed information about its physical properties, composition, and characteristics. Use this tool when you need comprehensive data about planets, moons, stars, asteroids, or other celestial bodies.",
        "input_schema": {
            "type": "object",
            "properties": {
                "object_name": {"type": "string", "description": "The name or designation of the celestial object (e.g., 'Mars', 'Europa', 'Proxima Centauri b')"},
                "object_type": {"type": "string", "enum": ["planet", "dwarf_planet", "moon", "asteroid", "comet", "star", "exoplanet", "nebula", "galaxy"], "description": "The classification of the celestial object"},
                "analysis_depth": {"type": "string", "enum": ["basic", "standard", "comprehensive"], "description": "Level of detail for the analysis"},
                "include_moons": {"type": "boolean", "description": "Whether to include information about the object's natural satellites"},
                "coordinate_system": {"type": "string", "enum": ["ecliptic", "equatorial", "galactic"], "description": "The coordinate system to use for positional data"},
                "epoch": {"type": "string", "description": "The reference epoch for orbital elements (e.g., 'J2000')"},
                "output_units": {
                    "type": "object",
                    "properties": {
                        "mass": {"type": "string", "enum": ["kg", "earth_masses", "solar_masses", "jupiter_masses"]},
                        "distance": {"type": "string", "enum": ["km", "au", "light_years", "parsecs"]},
                        "temperature": {"type": "string", "enum": ["kelvin", "celsius", "fahrenheit"]}
                    }
                }
            },
            "required": ["object_name", "object_type"]
        }
    },
    {
        "name": "calculate_orbital_mechanics",
        "description": "Performs orbital mechanics calculations including trajectory planning, orbital transfers, gravitational interactions, and mission feasibility assessments. Supports both two-body and n-body calculations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "calculation_type": {"type": "string", "enum": ["hohmann_transfer", "gravity_assist", "orbital_period", "escape_velocity", "delta_v", "sphere_of_influence", "lagrange_points", "orbital_decay"]},
                "origin_body": {"type": "string", "description": "The starting celestial body or orbit"},
                "destination_body": {"type": "string", "description": "The target celestial body or orbit"},
                "orbital_elements": {
                    "type": "object",
                    "properties": {
                        "semi_major_axis": {"type": "number"},
                        "eccentricity": {"type": "number"},
                        "inclination": {"type": "number"},
                        "longitude_ascending_node": {"type": "number"},
                        "argument_periapsis": {"type": "number"},
                        "true_anomaly": {"type": "number"}
                    }
                },
                "spacecraft_mass": {"type": "number", "description": "Mass of the spacecraft in kilograms"},
                "propulsion_system": {
                    "type": "object",
                    "properties": {
                        "type": {"type": "string", "enum": ["chemical", "ion", "nuclear_thermal", "solar_sail"]},
                        "specific_impulse": {"type": "number"},
                        "thrust": {"type": "number"}
                    }
                },
                "launch_window": {
                    "type": "object",
                    "properties": {
                        "start_date": {"type": "string"},
                        "end_date": {"type": "string"},
                        "optimization_criteria": {"type": "string", "enum": ["minimum_delta_v", "minimum_time", "balanced"]}
                    }
                }
            },
            "required": ["calculation_type", "origin_body"]
        }
    },
    {
        "name": "search_exoplanet_database",
        "description": "Searches comprehensive exoplanet databases including NASA Exoplanet Archive and EU Exoplanet Catalogue. Returns detailed information about confirmed exoplanets, host stars, detection methods, and habitability assessments.",
        "input_schema": {
            "type": "object",
            "properties": {
                "search_mode": {"type": "string", "enum": ["by_name", "by_star", "by_criteria", "by_region"]},
                "planet_name": {"type": "string"},
                "host_star": {"type": "string"},
                "discovery_method": {"type": "string", "enum": ["transit", "radial_velocity", "direct_imaging", "microlensing", "timing", "astrometry"]},
                "mass_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "radius_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "orbital_period_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "habitable_zone_only": {"type": "boolean"},
                "equilibrium_temperature_range": {"type": "object", "properties": {"min": {"type": "number"}, "max": {"type": "number"}}},
                "stellar_type": {"type": "string", "enum": ["O", "B", "A", "F", "G", "K", "M"]},
                "distance_from_earth": {"type": "object", "properties": {"max_light_years": {"type": "number"}}},
                "sort_by": {"type": "string", "enum": ["distance", "mass", "radius", "discovery_date", "habitability_score"]},
                "max_results": {"type": "integer"}
            },
            "required": ["search_mode"]
        }
    },
    {
        "name": "simulate_mission_trajectory",
        "description": "Simulates complete spacecraft mission trajectories from launch to destination arrival. Includes multi-body gravitational effects, maneuver planning, and mission phase modeling.",
        "input_schema": {
            "type": "object",
            "properties": {
                "mission_name": {"type": "string"},
                "mission_type": {"type": "string", "enum": ["flyby", "orbiter", "lander", "sample_return", "crewed", "telescope"]},
                "launch_site": {"type": "string", "enum": ["kennedy_space_center", "vandenberg", "baikonur", "kourou", "tanegashima", "satish_dhawan", "cape_canaveral"]},
                "launch_date": {"type": "string"},
                "target_body": {"type": "string"},
                "gravity_assists": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "body": {"type": "string"},
                            "approach_altitude": {"type": "number"},
                            "expected_date": {"type": "string"}
                        }
                    }
                },
                "spacecraft_configuration": {
                    "type": "object",
                    "properties": {
                        "dry_mass": {"type": "number"},
                        "fuel_mass": {"type": "number"},
                        "power_system": {"type": "string", "enum": ["solar_panels", "rtg", "nuclear_reactor", "battery"]},
                        "communication_band": {"type": "string", "enum": ["S_band", "X_band", "Ka_band", "optical"]}
                    }
                },
                "simulation_parameters": {
                    "type": "object",
                    "properties": {
                        "time_step": {"type": "number"},
                        "include_perturbations": {"type": "boolean"},
                        "include_solar_pressure": {"type": "boolean"},
                        "monte_carlo_runs": {"type": "integer"}
                    }
                },
                "output_format": {"type": "string", "enum": ["summary", "detailed", "trajectory_data", "visualization"]}
            },
            "required": ["mission_name", "mission_type", "launch_date", "target_body"]
        }
    }
]

QUESTION = "I want to plan a mission to Europa. Can you help me understand if it's feasible and what tools you'd use?"

print(f"Defined {len(TOOLS_LIST)} tools")

## 1. Converse API — Tool Definition Caching

Tools use `toolSpec` with `inputSchema.json` for the schema. The `cachePoint` is a separate element **appended after all tool specs**:

```python
toolConfig = {
    "tools": [
        {"toolSpec": {"name": ..., "inputSchema": {"json": ...}}},
        {"cachePoint": {"type": "default"}}   # after all tools
    ]
}
```

In [ ]:
def build_converse_tools():
    tools = []
    for tool in TOOLS_LIST:
        tools.append({
            "toolSpec": {
                "name": tool["name"],
                "description": tool["description"],
                "inputSchema": {"json": tool["input_schema"]}
            }
        })
    tools.append({"cachePoint": {"type": "default"}})
    return tools


def converse_tools_cached():
    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": QUESTION}]}],
        toolConfig={"tools": build_converse_tools()},
        inferenceConfig={"maxTokens": 512}
    )
    return response["usage"]

# Request 1 — cache write
usage1 = converse_tools_cached()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_tools_cached()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## 2. ConverseStream API — Tool Definition Caching with Streaming

Same tool caching pattern with streaming output.

In [ ]:
def converse_stream_tools_cached():
    response = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": QUESTION}]}],
        toolConfig={"tools": build_converse_tools()},
        inferenceConfig={"maxTokens": 512}
    )

    text = ""
    usage = {}
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"]["delta"]
            if "text" in delta:
                text += delta["text"]
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})

    return usage

# Request 1 — cache write
usage1 = converse_stream_tools_cached()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_stream_tools_cached()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## Conclusion

Tool definition caching is ideal for **agentic workflows** where the same set of tools is used across many requests with varying user queries. By caching the tool schemas, you avoid reprocessing thousands of tokens of schema definitions on every turn.

The `cachePoint` syntax in `toolConfig` is model-agnostic — works with any Bedrock model that supports prompt caching.

For Anthropic-specific InvokeModel syntax (`cache_control` on the last tool), see [invoke_model_api/anthropic/](../../invoke_model_api/anthropic/).

For more details, see the [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).